In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# تحديد خيارات Sampler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="إصدارات الحزم">

الكود في هذه الصفحة تم تطويره باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
يمكنك استخدام الخيارات لتخصيص الأداة البدائية Sampler. يركز هذا القسم على كيفية تحديد خيارات الأداة البدائية في Qiskit Runtime. بينما تكون واجهة طريقة `run()` للأدوات البدائية مشتركة عبر جميع التطبيقات، إلا أن خياراتها ليست كذلك. راجع مراجع API المقابلة للمزيد من المعلومات حول خيارات [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) و [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## ضبط خيارات Sampler
يمكنك ضبط الخيارات عند تهيئة Sampler، أو بعد تهيئته، أو يمكنك تحديث الخيارات بعد تهيئة Sampler. للتعليمات حول استخدام هذه التقنيات، انظر موضوع [مقدمة الخيارات](/guides/runtime-options-overview#options-precedence).

بالإضافة إلى ذلك، يمكنك ضبط قيمة `shots` في طريقة `run()`، كما هو موضح في القسم التالي.
<span id="run-method"></span>
### طريقة Run()
القيم الوحيدة التي يمكنك تمريرها إلى `run()` هي تلك المحددة في الواجهة. أي `shots`. يؤدي هذا إلى الكتابة فوق أي قيمة محددة لـ `default_shots` للتشغيل الحالي.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### حالات خاصة
<span id="shots"></span>
#### Shots

تقبل طريقة `SamplerV2.run` وسيطتين: قائمة من PUBs، يمكن لكل منها تحديد قيمة shots خاصة بها، ووسيطة keyword للـ shots. قيم shots هذه جزء من واجهة تنفيذ Sampler، وهي مستقلة عن خيارات Runtime Sampler. وهي تأخذ الأولوية على أي قيم محددة كخيارات للتوافق مع تجريد Sampler.

ومع ذلك، إذا لم يتم تحديد `shots` من قِبل أي PUB أو في وسيطة keyword للـ run (أو إذا كانت جميعها `None`)، فيتم استخدام قيمة shots من الخيارات، وأبرزها `default_shots`.

للتلخيص، هذا هو ترتيب الأولوية لتحديد shots في Sampler، لأي PUB بعينه:

1. إذا حدد PUB shots، استخدم تلك القيمة.
2. إذا تم تحديد وسيطة keyword `shots` في `run`، استخدم تلك القيمة.
4. إذا كان `twirling` مفعّلاً (True بشكل افتراضي)، فيُستخدم حاصل ضرب `num_randomizations` و `shots_per_randomization`، كما هو محدد في [خيارات `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. إذا تم تحديد `sampler.options.default_shots`، استخدم تلك القيمة.

وبالتالي، إذا تم تحديد shots في جميع الأماكن الممكنة، يُستخدم الأعلى أولوية (shots المحددة في PUB).

مثال: